# Act 1 — Cloud Storage: classes and locations are orthogonal

Cloud Storage (GCS) is the object store. Buckets hold objects (blobs); each bucket has a globally unique name and lives in a location. Two axes shape every bucket decision:

- **Storage class** — how often you'll access the data, which trades hot-access price for cold-access price.
- **Location** — single-region, dual-region, or multi-region, which trades replication breadth for cost.

**The crucial bit:** these two axes are *orthogonal*. You pick a class **and** a location independently. A `nearline` bucket in `us-central1` is different from a `nearline` bucket in the `us` multi-region. People constantly conflate them; getting them clear pays off across this whole chapter.

## Storage classes

| Class | Min storage duration | At-rest price | Read price | Use case |
|---|---|---|---|---|
| **Standard** | None | Highest | Cheapest | Hot data: serving, current builds, active datasets |
| **Nearline** | 30 days | Lower | Higher | Accessed once a month or less: backups, monthly reports |
| **Coldline** | 90 days | Lower still | Higher still | Quarterly access: DR copies, long-term archives in active rotation |
| **Archive** | 365 days | Lowest | Highest | Once-a-year-or-less: regulatory archive, finalised cold data |

**Minimum storage duration** is the trap. If you write a 1 TB object to Archive and delete it after a week, you still pay the full year of at-rest cost. Match the class to the *real* access cadence, not the *hoped-for* one.

**Class changes** can be done object-by-object (`gcloud storage objects update --storage-class`) or driven by lifecycle policy (next section). The class is per-object, *not* per-bucket — a bucket's class is just the default for new objects.

## Location types

From notebook 01 — three location types, each a different replication story:

| Location type | Examples | Replication | Availability SLA |
|---|---|---|---|
| **Region** | `us-central1`, `europe-west1` | Multi-zone within one region | 99.9% (Standard) |
| **Dual-region** | `nam4` (Iowa + S.Carolina), `eur4` | Synchronous across two named regions | 99.95% |
| **Multi-region** | `us`, `eu`, `asia` | Async across multiple regions inside the location | 99.95% |

**Dual-region is the most underrated choice.** You get *synchronous* writes to two regions — turbo-charged Recovery Point Objective with no "eventually consistent" gotcha — and the regions are named, so you know exactly where your data lives. Multi-region is convenient but the underlying replica set is opaque; dual-region gives you explicit answers for compliance questions.

**Pick location at create time only.** You cannot change a bucket's location once created — you'd `gcloud storage cp` to a new bucket and rename in your application. Get it right the first time.

# Act 2 — GCS access and lifecycle

Once you've placed objects, four orthogonal controls shape what happens to them: lifecycle, versioning, retention, and access. Mix-and-match these to fit your workload; each one is independently configurable.

## Lifecycle rules — automated class transitions and deletion

A **lifecycle rule** on a bucket says: under condition X, take action Y on the object. Two common shapes:

```
IF object age > 30 days       → SetStorageClass to NEARLINE
IF object age > 365 days       → SetStorageClass to ARCHIVE
IF object age > 7 years        → Delete
```

Conditions you can match on:

- `age` in days
- `createdBefore` / `customTimeBefore` — absolute dates
- `matchesStorageClass` — only act on objects currently in this class
- `matchesPrefix` / `matchesSuffix` — filename patterns
- `noncurrentTimeBefore` — for versioned buckets, when the version became non-current

Actions are either `Delete` or `SetStorageClass`. Lifecycle rules run once a day; expect ~24 h between hitting the condition and seeing the action.

**The dominant pattern**: write to Standard, downgrade to Nearline after 30 days, Archive after 90, delete after retention. Implement that once and you'll save more on a year of storage cost than any other single optimisation in this chapter.

## Versioning, retention, and Bucket Lock

**Object versioning.** Off by default. Turn it on and every overwrite or delete keeps the prior version as a *non-current* object. Recovery from accidental deletes becomes trivial — but storage bill goes up unless you pair versioning with a lifecycle rule that ages out non-current versions.

**Retention policy.** A bucket-level minimum lifetime. Set `retentionPolicy: 7y` and no object can be deleted within 7 years of creation (or modification). Used for regulatory compliance.

**Bucket Lock.** Locks the retention policy itself. After locking, the retention period **cannot be shortened or removed** — by you, your project owner, or anyone short of Google support with a documented legal process. This is the WORM (Write Once Read Many) compliance pattern, equivalent to S3 Object Lock or Azure immutable blob storage.

The stack to remember: versioning protects against accidental damage, retention enforces minimum lifetime, Bucket Lock makes the retention itself immutable. Layered correctly, the chain is hard to break even by an attacker who steals owner credentials.

## Uniform vs fine-grained access

GCS has two access control models living in the same product, and the choice is a one-time bucket setting:

- **Uniform bucket-level access (UBLA)** — only IAM applies. Object ACLs are disabled. The modern default; enforced by an Org Policy constraint in most enterprises.
- **Fine-grained access** — legacy, per-object ACLs *in addition to* IAM. ACLs predate GCS IAM and exist for backward compatibility.

**Always uniform for new buckets.** Object ACLs are a long-tail security headache (the ACL on one object can override expectations set by the bucket's IAM policy). Uniform makes the access surface a single readable IAM policy.

**Signed URLs and signed policy documents** are the answer when you need to grant short-lived, bounded access to a specific principal without giving them IAM. A **signed URL** is a time-limited URL that grants read or write on one object; you generate it from a service account's credentials. **Signed policy documents** generalise to multi-file uploads with conditions on size and content type. Both are the GCS analogues of S3 pre-signed URLs.

**Requester pays** flips the billing for downloads — the *requester*, not the bucket owner, is charged for egress. Used by public-good datasets (the bucket owner publishes; downloaders pay their own transfer).

## Code — generating a signed URL

Generate a signed URL for a one-hour upload from a service account.

In [ ]:
from datetime import timedelta
from google.cloud import storage

client = storage.Client()
bucket = client.bucket("acme-uploads")
blob = bucket.blob("user-42/profile.jpg")

url = blob.generate_signed_url(
    version="v4",
    expiration=timedelta(hours=1),
    method="PUT",
    content_type="image/jpeg",
)
print(url)
# Caller can now PUT a JPEG to this URL for the next hour; no GCP creds needed.

# Act 3 — Encryption

All Cloud Storage data is encrypted at rest, full stop. The question is who holds the keys. Three options, escalating in operational burden and customer control.

## Three encryption modes

| Mode | Key held by | Setup | Use when |
|---|---|---|---|
| **Google-managed** | Google | Default, automatic | Default — pick this unless you have a reason not to |
| **CMEK (Customer-Managed Encryption Keys)** | You, in Cloud KMS | Create a KMS key, grant the GCS service agent `roles/cloudkms.cryptoKeyEncrypterDecrypter` | Compliance asks for customer-managed keys; you want central rotation / disable / destroy |
| **CSEK (Customer-Supplied Encryption Keys)** | You, outside GCP | Pass an `x-goog-encryption-key` header per request | Niche — regulatory rules forbid storing key material in GCP at all |

**CMEK is the realistic enterprise default.** Cloud KMS handles rotation, audit logs every key use, and lets you disable the key to render objects unreadable in seconds (a useful incident-response lever). The price is small: every object operation does a KMS API call.

**CSEK is rare.** You ship the raw key with every request; lose it and the data is unrecoverable. Useful only when the regulator literally says "the cloud provider must never possess our keys."

**Cloud HSM** sits underneath CMEK as a key-storage option: keys are generated and stored on FIPS 140-2 Level 3 hardware. Same KMS API, different backend. Notebook 11 covers KMS in detail.

# Act 4 — Block and file storage

GCS handles objects. For block storage (a virtual hard drive attached to a VM) and file storage (an NFS or SMB share), GCP has separate products with their own shape. Most chapters won't need this in depth; the names and trade-offs are enough.

## Persistent Disk and Hyperdisk

**Persistent Disk (PD)** is the original block storage — attached to a Compute Engine VM as a boot or data disk. Four types span the cost-performance curve:

| Type | Backed by | Performance shape | When |
|---|---|---|---|
| **Standard (pd-standard)** | HDD | Cheapest, lowest IOPS | Logs, archives, dev/test |
| **Balanced (pd-balanced)** | SSD | Good IOPS, lower cost than full SSD | General-purpose default |
| **SSD (pd-ssd)** | SSD | High IOPS | Databases, latency-sensitive |
| **Extreme (pd-extreme)** | SSD with provisioned IOPS | Highest IOPS, you tune IOPS independently of size | High-end OLTP |

**Hyperdisk** is the modern replacement, rolling out across machine families. Three variants — Throughput, Balanced, Extreme — and the headline change is that **performance is configured independently of capacity**. With PD, doubling IOPS means doubling disk size; with Hyperdisk, you dial IOPS, throughput, and capacity separately. New designs in 2026+ default to Hyperdisk.

**Local SSD** is a different animal — physically attached to the VM host, not a network-attached block device. Lowest possible latency, highest IOPS — but **ephemeral**: the SSD's contents are lost when the VM stops, terminates, or migrates. Use for scratch space, distributed caches, and temporary database state. Never use for the only copy of anything.

**Regional Persistent Disk** synchronously replicates a PD across two zones in a region — used as the storage layer for regional Cloud SQL HA and any DIY zonal-failover scheme.

## Filestore — managed NFS

**Filestore** is GCP's managed NFS file share. Mounted by NFSv3 clients (VMs, GKE PersistentVolumes) like any traditional file server.

| Tier | Performance | Use case |
|---|---|---|
| **Basic HDD / Basic SSD** | Modest | Dev/test, low-throughput legacy apps |
| **Zonal** | Higher IOPS, single zone | Production GKE workloads, single-zone HPC |
| **Enterprise** | High availability across zones in a region | Production multi-zone shared storage |
| **Regional** | Multi-zone HA, high performance | Mission-critical shared file workloads |

Filestore exists mostly because legacy applications expect a POSIX file system. New designs should prefer Cloud Storage with FUSE (`gcsfuse`) or object semantics — but you'll meet Filestore in lifts-and-shifts.

## Moving data in — Transfer Service and Transfer Appliance

Three ways to get bulk data into GCP, scaling by data size:

| Tool | What it does | Use when |
|---|---|---|
| **`gcloud storage cp` / `gsutil cp`** | Multi-threaded copy from your shell | Megabytes to single-digit terabytes, one-shot |
| **Storage Transfer Service** | Server-side scheduled transfer | Recurring or large transfers from S3, Azure Blob, HTTP/HTTPS, on-prem POSIX |
| **Transfer Appliance** | Rackable hardware Google ships you, you fill, ship back | Petabyte-scale; bandwidth would take months |

**Storage Transfer Service is the workhorse.** Cross-cloud migration from S3, recurring on-prem ingest, periodic sync between buckets — all in this one product. It handles checksumming, retries, and (with Premium tier) sub-hourly schedules.

**Transfer Appliance** is genuinely a shipped box (TA40 is ~480 TB, TA300 is ~300 TB). Used when shipping a hard drive over FedEx is faster than transferring over your link. The Snowball / Data Box analogue.

## What carries into later chapters

GCS is the most-touched storage primitive in the rest of this course. BigQuery reads/writes GCS (notebook 09). Dataflow reads/writes GCS. Cloud Build pushes artifacts to GCS. Cloud Logging exports go to GCS via Log Router (notebook 12).

Three habits to carry forward:

- **Class and location are independent.** Pick both deliberately.
- **Lifecycle rules pay for themselves within months.** Standard → Nearline → Archive → Delete is rarely wrong.
- **CMEK on regulated buckets, Google-managed everywhere else.** Don't over-engineer encryption for buckets that hold non-sensitive artifacts.

Notebook 06 is networking — VPCs, subnets, firewall rules, and how Cloud Run, GCE, and GCS actually talk to each other privately.